# ForceLorentz — `force_lorentz`

`Postraitement_fields` · ASN 28 validation form (TRUST-LiMe).



## Introduction

Validation of the **differential** post-treatment fields of the standalone
magnetic (induction) equation:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

This notebook runs **one and only one test case**: `force_lorentz`. Per the
TRUST validation-form layout the other `Postraitement_fields` cases live in
their own sibling folders.

## What it validates

$$
\mathbf{F} = \mathbf{J} \times \mathbf{b} = (\nabla \times \mathbf{b}) \times \mathbf{b} / \mu_0
$$
Exactly linear field $b = (-y, x)$: $w_z = 2$, so with $\mu_0 = 1$:
$F = (-w_z b_y,\ w_z b_x) = (-2x,\ -2y)$ in every cell. At the probed cell
centre $(0.45, 0.55)$: $F = (-0.9, -1.1)$. (In 3D the Lorentz force is a
full vector; here $b$ is planar so $w$ is out-of-plane and $F$ is in-plane.)


## Contract checked

| # | Quantity | Formula |
|---|---|---|
| 1 | Lorentz force density | $F = (\nabla\times b)\times b / \mu_0$ |

All these fields are built from the module-reconstructed element gradient
$\nabla b$ (`gradient_b_elements`, Green-Gauss — the kernel has no
element-field gradient factory on PolyMAC). Vector model required: the
scalar model is rejected with a clean error by `creer_champ`.



In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()



## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.



In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case`."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_son_all(case, sonde):
    """Like read_son but returns every entry (components of a vector field)."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    return [(e, np.asarray(son.getValues(e)[1], dtype=float)) for e in son.getEntries()]

print("BUILD =", BUILD)



## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).



In [ ]:
import pathlib, shutil

from trustutils import run

NB_PROCS = 2

run.reset()
case = run.addCase(".", "force_lorentz.data", nbProcs=NB_PROCS)
case.partition()          # trust -partition force_lorentz.data 2 -> PAR_force_lorentz.data + DOM_000*.Zones
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob(f"{case.dataFileName_}_*.son"):
    shutil.copy2(son, build / son.name)



## Checks


In [ ]:
entries = read_son_all(case.dataFileName_, "SONDE_FL")
for name, v in entries:
    print(name, "=", v[-1])
comps = [v[-1] for name, v in entries]

# b = (-y, x) is exactly linear => F = (-2 x, -2 y) with mu0 = 1.
expected = np.array([-2 * 0.45, -2 * 0.55])
print(f"force_lorentz (probe) : ({comps[0]:.6g}, {comps[1]:.6g})  (expected {expected})")
err = np.max(np.abs(np.array(comps) - expected)) / np.max(np.abs(expected))
print(f"relative error        : {err:.3e}")
assert err < 1e-12, f"force_lorentz wrong: {comps}"
print("OK: force_lorentz = (curl b) x b / mu0.")


## Computer Performance


In [ ]:
run.tablePerf()

